# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

* Built a clean numerical feature vector using `word_count`, `content_age_days`, `impressions_90d`, `avg_position`, `ctr`, and `search_volume`. Missing values in numerical features are imputed using baseline median and zero filling to preserve full row availability without data drops.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import duckdb
import os, sys

# Setup repository path safely for Google Colab
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 1. Define Selected Feature Columns
feature_cols = [
    'word_count',
    'content_age_days',
    'impressions_90d',
    'avg_position',
    'ctr',
    'search_volume'
]

# 2. Build Feature Matrix X
X = df[feature_cols].copy()

# 3. Handle Missing Values (Imputation)
X['word_count'] = X['word_count'].fillna(X['word_count'].median())
X['impressions_90d'] = X['impressions_90d'].fillna(0)
X['ctr'] = X['ctr'].fillna(0.0)
X['search_volume'] = X['search_volume'].fillna(X['search_volume'].median())

# 4. Formulate Binary Target Label y (1 = declining, 0 = non-declining)
y = (df['trend_direction'] == 'down').astype(int)

print(f"Feature Vector Matrix Shape (X): {X.shape}")
print(f"Target Vector Shape (y): {y.shape}")
print(f"\nTarget Class Distribution (1 = Declining):\n{y.value_counts(normalize=True)}")
X.head()

Feature Vector Matrix Shape (X): (30000, 6)
Target Vector Shape (y): (30000,)

Target Class Distribution (1 = Declining):
trend_direction
1    0.542067
0    0.457933
Name: proportion, dtype: float64


,word_count,content_age_days,impressions_90d,avg_position,ctr,search_volume
0,3221.0,187,3803,10.6,0.76,10.0
1,2481.0,445,15320,20.3,0.05,90.0
2,3515.0,141,12581,36.5,0.09,0.0
3,2877.0,463,11751,6.2,0.49,10.0
4,2803.0,263,19140,44.0,0.13,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

* `word_count`: Total word length of the page content. Missing values imputed using dataset median. Fully available pre-prediction.
* `content_age_days`: Age of the content page in days. No missing values. Fully available pre-prediction.
* `impressions_90d`: Total Google SERP impressions over past 90 days. Missing values imputed with 0. Fully available pre-prediction.
* `avg_position`: Mean ranking position in Google search results. Fully available pre-prediction.
* `ctr`: Historical click-through rate. Missing values imputed with 0.0. Fully available pre-prediction.
* `search_volume`: Monthly search demand volume. Missing values imputed with dataset median. Fully available pre-prediction.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Summary Statistics & Missingness Audit for Selected Features
feature_summary = X.describe().T[['mean', 'std', 'min', '50%', 'max']]
feature_summary['missing_count'] = df[feature_cols].isnull().sum()
feature_summary['missing_pct'] = (df[feature_cols].isnull().sum() / len(df)) * 100

print("Feature Notes & Imputation Audit Table:")
feature_summary.round(2)

Feature Notes & Imputation Audit Table:


,mean,std,min,50%,max,missing_count,missing_pct
word_count,3048.54,1256.27,8.0,2877.00,9546.0,7699,25.66
content_age_days,256.17,132.71,90.0,236.00,564.0,0,0.00
impressions_90d,5200.37,16838.02,1.0,731.00,517715.0,0,0.00
avg_position,16.34,15.22,0.0,10.80,245.0,0,0.00
ctr,0.51,3.28,0.0,0.07,100.0,0,0.00
search_volume,146.63,1455.05,0.0,10.00,74000.0,2468,8.23


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

* Tested feature matrix $X$ against target $y$ for potential data leakage.
* Evaluated Pearson correlation between all selected numerical features and the binary target ($y$).
* Confirmed no feature exceeds a high correlation threshold ($\ge 0.85$). Highest observed correlation is moderate and reflects historical behavior rather than future target leakage.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Correlation check between features and target to detect feature leakage
leakage_audit = X.copy()
leakage_audit['target_declining'] = y

# Calculate correlation with target
corr_matrix = leakage_audit.corr()['target_declining'].drop('target_declining')
leakage_summary = pd.DataFrame({
    'Pearson Correlation with Target': corr_matrix,
    'Absolute Correlation': corr_matrix.abs(),
    'Leakage Risk Level': corr_matrix.abs().apply(
        lambda r: 'CRITICAL (Leakage)' if r >= 0.85 else ('MODERATE' if r >= 0.3 else 'LOW')
    )
}).sort_values(by='Absolute Correlation', ascending=False)

print("--- Leakage Audit: Feature Correlation with Target ---")
leakage_summary.round(4)

--- Leakage Audit: Feature Correlation with Target ---


,Pearson Correlation with Target,Absolute Correlation,Leakage Risk Level
content_age_days,-0.1639,0.1639,LOW
word_count,0.0843,0.0843,LOW
ctr,-0.0619,0.0619,LOW
avg_position,-0.0290,0.0290,LOW
impressions_90d,-0.0182,0.0182,LOW
search_volume,-0.0141,0.0141,LOW


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* `trend_direction`: Raw string label source for target $y$; using it causes 100% target leakage.
* `trend_pct`: Direct percentage change metric that explicitly reveals the target trajectory.
* `content_id` / `client_id`: Unique database primary keys that carry no predictive domain signal.
* `impressions_last_30d` / `clicks_last_30d`: Short-term metrics that overlap temporally with 90-day aggregates.
* `provider_used` / `model_used`: Platform infrastructure metadata unrelated to content quality or search ranking performance.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify excluded columns and confirm they are not present in Feature Matrix X
excluded_fields = [
    'trend_direction', 'trend_pct', 'content_id', 'client_id',
    'impressions_last_30d', 'clicks_last_30d', 'provider_used', 'model_used'
]

# Audit check
all_excluded_clean = all(col not in X.columns for col in excluded_fields)

print("--- Excluded Fields Audit ---")
for field in excluded_fields:
    status = "REJECTED (Not in X)" if field not in X.columns else "ERROR (Present in X)"
    print(f"• {field}: {status}")

print(f"\nFinal Feature Matrix Leakage Guard Verdict: {'PASSED' if all_excluded_clean else 'FAILED'}")

--- Excluded Fields Audit ---
• trend_direction: REJECTED (Not in X)
• trend_pct: REJECTED (Not in X)
• content_id: REJECTED (Not in X)
• client_id: REJECTED (Not in X)
• impressions_last_30d: REJECTED (Not in X)
• clicks_last_30d: REJECTED (Not in X)
• provider_used: REJECTED (Not in X)
• model_used: REJECTED (Not in X)

Final Feature Matrix Leakage Guard Verdict: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.